# CVD Death Rate Forecasting — Exploratory Data Analysis

**Purpose:** Spike work to answer four structural questions that determine every downstream engineering decision.

**Questions:**
1. How many data points per country? (constrains per-country modeling)
2. Do countries share similar trajectory shapes? (global vs. per-country)
3. How strong is temporal autocorrelation? (informs lag features)
4. Within-country vs. between-country variance (determines fixed effects importance)

**Output:** A written conclusion cell at the end — the design decision record.

In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.graphics.tsaplots import plot_acf
from tqdm import tqdm
import warnings
import io

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.dpi": 120, "font.size": 11, "axes.titlesize": 13})

# World Bank income group sample countries.
# One row per group — used throughout for plots and ACF analysis.
INCOME_GROUP_SAMPLES = {
    "High income":         ["United States", "Germany", "Japan"],
    "Upper middle income": ["China", "Brazil", "Mexico"],
    "Lower middle income": ["India", "Nigeria", "Pakistan"],
    "Low income":          ["Ethiopia", "Mali", "Chad"],
}
SAMPLE_COUNTRIES = [c for group in INCOME_GROUP_SAMPLES.values() for c in group]
print("Setup complete.")

## 1. Pull raw data from OWID

OWID exposes datasets as CSVs on GitHub. We pull the IHME CVD death rate dataset directly.

**Why not the Grapher API?** It is undocumented and unstable. The GitHub CSV is what the OWID data team maintains and is the canonical source.

**Metric:** Age-standardised CVD death rate per 100,000 population. Age-standardisation is critical — it controls for aging populations so we measure actual cardiovascular health, not demographics.

In [ ]:
OWID_CVD_URL = (
    "https://raw.githubusercontent.com/owid/owid-datasets/master/"
    "datasets/Cardiovascular%20disease%20death%20rates%20-%20IHME%20(2019)/"
    "Cardiovascular%20disease%20death%20rates%20-%20IHME%20(2019).csv"
)

print("Fetching OWID CVD dataset...")
response = requests.get(OWID_CVD_URL, timeout=30)
response.raise_for_status()

df_raw = pd.read_csv(io.StringIO(response.text))
print(f"Raw shape: {df_raw.shape}")
print(f"Columns: {df_raw.columns.tolist()}")
print(f"Year range: {df_raw['Year'].min()} - {df_raw['Year'].max()}")
print(f"Unique entities: {df_raw['Entity'].nunique()}")

In [ ]:
# Always look at raw data before touching it.
# Surprises live here: column names, nulls, aggregates mixed with countries.
df_raw.head(10)

In [ ]:
df_raw.info()
print("
Null counts:")
print(df_raw.isnull().sum())

In [ ]:
df_raw.describe()

## 2. Standardise column names

Normalise to snake_case before any analysis. This is a habit that pays off in dbt — your notebook column names and dbt model column names should match.

In [ ]:
cols = df_raw.columns.tolist()
rename_map = {cols[0]: "country", cols[1]: "country_code", cols[2]: "year"}

for i, col in enumerate(cols[3:], start=3):
    clean = (
        col.lower()
        .replace(" ", "_").replace("-", "_")
        .replace("(", "").replace(")", "").replace(",", "")
    )
    rename_map[col] = f"metric_{i-2}_{clean[:40]}"

df = df_raw.rename(columns=rename_map)
print("Renamed columns:")
for old, new in rename_map.items():
    print(f"  {repr(old):60s} -> {repr(new)}")

In [ ]:
metric_cols = [c for c in df.columns if c.startswith("metric_")]
print("Metric columns:")
for col in metric_cols:
    print(f"  {col}  |  non-null: {df[col].notna().sum()}")

# Set this to the primary CVD death rate column after inspecting above.
TARGET_COL = metric_cols[0]
print(f"
Using target column: {TARGET_COL}")

## 3. Separate countries from aggregate entities

OWID mixes true countries with regional aggregates ("World", "Sub-Saharan Africa", etc.).
These must be separated before modeling — they are not independent observations.

**Strategy:** Rows without a valid 3-letter ISO country code are aggregates.

In [ ]:
# ISO 3166-1 alpha-3 codes are exactly 3 uppercase letters.
# Aggregates use codes like "OWID_WRL" or have no code at all.
mask_country = df["country_code"].str.match(r"^[A-Z]{3}$", na=False)

df_countries  = df[mask_country].copy()
df_aggregates = df[~mask_country].copy()

print(f"Country rows:   {len(df_countries):,}")
print(f"Aggregate rows: {len(df_aggregates):,}")
print(f"
Aggregates found:")
print(df_aggregates["country"].unique())

In [ ]:
n_countries = df_countries["country"].nunique()
print(f"Unique countries: {n_countries}")

year_coverage = (
    df_countries.groupby("country")["year"]
    .agg(["min", "max", "count"])
    .rename(columns={"min": "first_year", "max": "last_year", "count": "n_years"})
    .sort_values("n_years")
)
print(f"
Year coverage summary:")
print(year_coverage.describe())

## 4. EDA Q1 — How many data points per country?

This is the most important structural constraint on modeling.
- Per-country XGBoost needs enough rows to generalise.
- Per-country Prophet needs enough for trend estimation.
- If most countries have <30 rows, per-country modeling is unreliable.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of rows per country
ax = axes[0]
year_coverage["n_years"].hist(bins=30, ax=ax, color="steelblue", edgecolor="white", linewidth=0.5)
ax.axvline(year_coverage["n_years"].median(), color="firebrick", linestyle="--", linewidth=1.5,
           label=f'Median: {year_coverage["n_years"].median():.0f} years')
ax.set_xlabel("Number of years of data")
ax.set_ylabel("Number of countries")
ax.set_title("Distribution of data points per country")
ax.legend()

# 20 countries with fewest data points
ax2 = axes[1]
bottom_20 = year_coverage.head(20)
ax2.barh(bottom_20.index, bottom_20["n_years"], color="steelblue", edgecolor="white", linewidth=0.3)
ax2.axvline(30, color="firebrick", linestyle="--", linewidth=1.2, label="30-row threshold")
ax2.set_xlabel("Years of data")
ax2.set_title("20 countries with fewest data points")
ax2.legend()

plt.tight_layout()
plt.savefig("eda_q1_datapoints_per_country.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Countries with < 20 years: {(year_coverage['n_years'] < 20).sum()}")
print(f"Countries with < 30 years: {(year_coverage['n_years'] < 30).sum()}")
print(f"Countries with >= 30 years: {(year_coverage['n_years'] >= 30).sum()}")

## 5. EDA Q2 — Do countries share similar trajectory shapes?

Plot CVD rate over time for sample countries — one per income group.
If trajectory *shapes* are similar (just at different levels), a global model works.
If shapes are qualitatively different, country groupings are needed.

In [ ]:
GROUP_COLORS = {
    "High income":         "#2166ac",
    "Upper middle income": "#4dac26",
    "Lower middle income": "#d6604d",
    "Low income":          "#8e0152",
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True)
axes = axes.flatten()

for ax, (group, countries) in zip(axes, INCOME_GROUP_SAMPLES.items()):
    color = GROUP_COLORS[group]
    for i, country in enumerate(countries):
        subset = df_countries[df_countries["country"] == country][["year", TARGET_COL]].dropna()
        if subset.empty:
            print(f"  WARNING: {country} not found — skipping")
            continue
        alpha = [1.0, 0.65, 0.4][i]
        ax.plot(subset["year"], subset[TARGET_COL], label=country, linewidth=2, color=color, alpha=alpha)
    ax.set_title(group, fontweight="bold", color=color)
    ax.set_ylabel("CVD death rate per 100k")
    ax.set_xlabel("Year")
    ax.legend(fontsize=9)

fig.suptitle("CVD Death Rate Trajectories by World Bank Income Group", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("eda_q2_trajectories_by_income_group.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Spaghetti plot — all countries overlaid.
# The point is to see overall spread and any outlier trajectories.

fig, ax = plt.subplots(figsize=(14, 6))

for country, grp in df_countries.groupby("country"):
    grp_sorted = grp.sort_values("year")
    ax.plot(grp_sorted["year"], grp_sorted[TARGET_COL],
            color="steelblue", alpha=0.07, linewidth=0.8)

# Overlay world aggregate for reference
world = df_aggregates[df_aggregates["country"].str.lower().str.contains("world", na=False)]
if not world.empty:
    world_s = world.sort_values("year")
    ax.plot(world_s["year"], world_s[TARGET_COL],
            color="firebrick", linewidth=2.5, label="World average", zorder=5)
    ax.legend()

ax.set_xlabel("Year")
ax.set_ylabel("CVD death rate per 100k")
ax.set_title("All countries — CVD death rate over time (spaghetti plot)")
plt.tight_layout()
plt.savefig("eda_q2_spaghetti.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. EDA Q3 — Temporal autocorrelation (ACF)

The ACF tells us: if I know this year's CVD rate, how much does that tell me about next year's?

High autocorrelation (ACF stays near 1.0 across many lags) means:
- Lag features will be the most powerful predictors in your model
- The series changes slowly — strong "memory"
- A naive model that just predicts "similar to last year" will look deceptively good on naive splits (this is why walk-forward CV matters)

In [ ]:
ACF_COUNTRIES = ["United States", "China", "India", "Ethiopia"]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for ax, country in zip(axes, ACF_COUNTRIES):
    series = (
        df_countries[df_countries["country"] == country]
        .sort_values("year")[TARGET_COL]
        .dropna()
        .reset_index(drop=True)
    )
    if len(series) < 10:
        ax.set_title(f"{country} — insufficient data")
        continue

    nlags = min(15, len(series) // 2 - 1)
    plot_acf(series, lags=nlags, ax=ax, zero=False, alpha=0.05)
    ax.set_title(f"ACF — {country} (n={len(series)} years)")
    ax.set_xlabel("Lag (years)")
    ax.set_ylabel("Autocorrelation")

fig.suptitle("Autocorrelation Function — CVD death rate by country", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("eda_q3_acf.png", dpi=150, bbox_inches="tight")
plt.show()

print("Lag-1 autocorrelation:")
for country in ACF_COUNTRIES:
    series = (
        df_countries[df_countries["country"] == country]
        .sort_values("year")[TARGET_COL].dropna()
    )
    if len(series) > 2:
        print(f"  {country:20s}: {series.autocorr(lag=1):.4f}")

## 7. EDA Q4 — Within-country vs. between-country variance

Variance decomposition: how much of the total variance in CVD rates is explained by *which country* you are in (between) vs. *how the country changed over time* (within)?

- Between >> Within → Country identity is a critical feature. Fixed effects or income group encoding will be high-value.
- Within >> Between → Temporal trend is the main signal. Lag features and rolling means dominate.

In [ ]:
df_clean = df_countries.dropna(subset=[TARGET_COL]).copy()
grand_mean = df_clean[TARGET_COL].mean()

country_means = df_clean.groupby("country")[TARGET_COL].mean()
between_var = np.average(
    (country_means - grand_mean) ** 2,
    weights=df_clean.groupby("country")[TARGET_COL].count()
)
within_var = df_clean.groupby("country")[TARGET_COL].var().mean()
total_var  = df_clean[TARGET_COL].var()

pct_between = between_var / total_var * 100
pct_within  = within_var  / total_var * 100

print("=" * 50)
print("Variance decomposition")
print("=" * 50)
print(f"Grand mean CVD rate:           {grand_mean:.1f} per 100k")
print(f"Total variance:                {total_var:.1f}")
print(f"Between-country variance:      {between_var:.1f}  ({pct_between:.1f}%)")
print(f"Within-country variance (avg): {within_var:.1f}  ({pct_within:.1f}%)")
print()
if pct_between > 60:
    print(">> Between-country dominates. Country identity is a critical feature.")
else:
    print(">> Within-country variance is substantial. Temporal trend matters a lot.")

In [ ]:
# Normalised trajectories: strip out between-country level differences, show shape only.
# If lines converge in shape, a global model can learn the shared pattern.

fig, ax = plt.subplots(figsize=(14, 6))

for country, grp in df_countries.groupby("country"):
    grp_s = grp.sort_values("year").dropna(subset=[TARGET_COL])
    if len(grp_s) < 5 or grp_s[TARGET_COL].iloc[0] == 0:
        continue
    normalised = grp_s[TARGET_COL] / grp_s[TARGET_COL].iloc[0]
    ax.plot(grp_s["year"], normalised, color="steelblue", alpha=0.06, linewidth=0.8)

ax.axhline(1.0, color="firebrick", linestyle="--", linewidth=1.2, label="Baseline (first year = 1.0)")
ax.set_xlabel("Year")
ax.set_ylabel("CVD rate relative to first observation")
ax.set_title("Normalised CVD trajectories — shape only, level stripped out")
ax.legend()
plt.tight_layout()
plt.savefig("eda_q4_normalised_trajectories.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Null analysis — what is actually missing?

Two kinds of missingness require different imputation strategies:
- **Structural**: data was never collected (common in low-income countries pre-2000) — forward fill or group mean imputation
- **Random**: isolated gaps in an otherwise complete series — interpolation

The null heatmap below tells us which pattern dominates.

In [ ]:
null_rate = df_countries[TARGET_COL].isnull().mean() * 100
print(f"Overall null rate: {null_rate:.1f}%")

country_null = (
    df_countries.groupby("country")[TARGET_COL]
    .apply(lambda x: x.isnull().mean() * 100)
    .sort_values(ascending=False)
)
print(f"
Countries with >50% nulls:")
high_null = country_null[country_null > 50]
print(high_null.to_string() if not high_null.empty else "  None")
print(f"
Countries with any nulls: {(country_null > 0).sum()}")

In [ ]:
# Null heatmap for sample countries.
# Black = missing, white = present.
# Nulls clustered at the top (early years) = structural missing.
# Scattered nulls = random missing.

pivot = (
    df_countries[df_countries["country"].isin(SAMPLE_COUNTRIES)]
    .pivot_table(index="year", columns="country", values=TARGET_COL, aggfunc="first")
)

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(
    pivot.isnull(), ax=ax,
    cmap=["white", "black"], cbar=False,
    linewidths=0.3, linecolor="lightgrey"
)
ax.set_title("Null pattern — sample countries (black = missing)")
ax.set_xlabel("")
ax.set_ylabel("Year")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("eda_null_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Check country name consistency across sources

The Gold layer will join OWID data with WHO GHO and World Bank data. Before building that join, we need to know whether country names are consistent. A single mismatch means a row silently drops from the joined dataset.

This cell checks whether our sample countries appear with the expected names in OWID. Any mismatches need a normalisation map in the Silver cleaning step.

In [ ]:
print("Checking sample country names against OWID data:")
owid_countries = set(df_countries["country"].unique())

for country in SAMPLE_COUNTRIES:
    found = country in owid_countries
    if not found:
        candidates = [c for c in owid_countries if country.lower()[:4] in c.lower()]
        hint = f"  -> Candidates: {candidates[:3]}" if candidates else ""
        print(f"  [MISSING] {country}{hint}")
    else:
        print(f"  [OK]      {country}")

print("
Any MISSING entries need entries in a country_name_map.py constants file.")
print("We will build this when we start the Silver cleaning layer.")

## 10. Save locally

We are not writing to S3 yet — that is the pipeline's job. We save locally so subsequent notebooks can load without re-fetching.

This also previews the Bronze → Silver pattern: always save raw first, then save cleaned separately. Never overwrite raw.

In [ ]:
import os
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

df_raw.to_parquet("data/raw/owid_cvd_raw.parquet", index=False)
df_countries.to_parquet("data/processed/owid_cvd_countries.parquet", index=False)

print(f"Saved raw:     data/raw/owid_cvd_raw.parquet           ({len(df_raw):,} rows)")
print(f"Saved cleaned: data/processed/owid_cvd_countries.parquet ({len(df_countries):,} rows)")
print("
Note: these are local spike files. Production writes go to S3 Bronze/Silver.")

## 11. CONCLUSION — Design decision record

**Fill this in after running all cells above. This becomes the permanent design record for the project.**

---

### Q1: Data points per country

_How many countries had >= 30 years of data? What does this mean for per-country modeling viability?_

### Q2: Trajectory shapes across income groups

_Did high-income countries show declining trends? What did low-income look like? Are shapes similar enough to support a global model, or do they diverge enough to need separate treatment?_

### Q3: Temporal autocorrelation

_What was the lag-1 autocorrelation for your sample countries? What lag depths should we engineer as features (lag-1, lag-3, lag-5)?_

### Q4: Within vs. between variance

_What % of total variance is between-country? Does this confirm that country identity / income group is a critical feature?_

### Modeling decision

_Based on the above — one global model, per-country models, or global model with country characteristics as features? Why?_

### Pipeline implications

_What features must the Gold layer include? What does the null analysis tell us about the Silver imputation strategy?_